In [233]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
import torch.functional as F
import torch.utils.data as data


In [234]:
# device = torch.device("cuda")
device = torch.device("cpu")
# set_seed(42)

In [235]:
target_discrete_intervals = torch.tensor([100000, 350000]) # 0 to mniej niż 100k$, 1 pomiędzy, 2 więcej
validation_percent = 0.3 # Ile w walidacyjnym

categorical_columns = ["HallwayType", "HeatingType", "AptManageType", "TimeToBusStop", "TimeToSubway", "SubwayStation"]
columns_to_drop = []

### Ładowanie danych


Warto zparametryzować batch_size

Pewnie są dodatkowe parametry w data.DataLoader(...) które mogą nam pomóc


In [236]:
def Get_Train_Data():
    train_data = pd.read_csv("train_data.csv")
    train_data = train_data.drop(columns=columns_to_drop)
    return train_data

def Get_Train_Values(train_data):
    train_categorical = pd.get_dummies(train_data[categorical_columns]).astype(int)
    train_numerical = train_data.drop(columns=categorical_columns)
    return train_numerical, train_categorical


In [237]:
def Get_Final_Test_Data():
    final_test_data = pd.read_csv("test_data.csv", index_col=None)
    final_test_data = final_test_data.drop(columns=columns_to_drop)

    return final_test_data

In [238]:
def Get_Final_Test_Values(final_test_data):
    final_test_categorical = pd.get_dummies(final_test_data[categorical_columns]).astype(int)
    final_test_numerical = final_test_data.drop(columns=categorical_columns)
    return final_test_numerical, final_test_categorical

In [239]:
def Get_Train_And_Validate_Loaders(train_numerical, train_categorical, size):
    train_indices = np.random.rand(size)>validation_percent
    print("Train data size: ", size)

    numerical_data = torch.from_numpy(train_numerical.values[train_indices, 1:]).float()
    categorical_data = torch.from_numpy(train_categorical.values[train_indices]).float()
    numerical_train_targets = torch.from_numpy(train_numerical.values[train_indices, 0]).float()
    discrete_train_targets = torch.bucketize(numerical_train_targets, target_discrete_intervals) # Dyskretne etykiety

    print(numerical_data.shape)
    print(categorical_data.shape)
    print(discrete_train_targets.shape)

    validate_numerical_data = torch.from_numpy(train_numerical.values[~train_indices, 1:]).float()
    validate_categorical_data = torch.from_numpy(train_categorical.values[~train_indices]).float()
    numerical_validate_targets = torch.from_numpy(train_numerical.values[~train_indices, 0]).float()
    discrete_validate_targets = torch.bucketize(numerical_validate_targets, target_discrete_intervals) # Dyskretne etykiety

    train_dataset = data.TensorDataset(numerical_data, categorical_data, discrete_train_targets)
    validate_dataset = data.TensorDataset(validate_numerical_data, validate_categorical_data, discrete_validate_targets)

    train_loader = data.DataLoader(train_dataset, batch_size=32, shuffle=True)
    validate_loader = data.DataLoader(validate_dataset, batch_size=32, shuffle=False)

    return train_loader, validate_loader

In [240]:
def Get_Final_Test_Loader(final_test_numerical, final_test_categorical, size):
    print("Final test data size: ", size)

    final_test_numerical_data = torch.from_numpy(final_test_numerical.values).float()
    final_test_categorical_data = torch.from_numpy(final_test_categorical.values).float()

    final_test_loader = data.TensorDataset(final_test_numerical_data, final_test_categorical_data)

    return final_test_loader

### Liczenie dokładności

In [241]:
def calc_accuracy(pred_targets, targets):
    accuracies = []
    for i in range(3):
        class_correct=(pred_targets.squeeze() == targets)[targets == i].sum() # tu zmieniłem żeby z tensorami działało
        accuracies.append(class_correct/(targets == i).sum())
    return(np.mean(accuracies))

In [242]:
def Get_Accuracy(model, data_loader):
    all_preds = torch.empty(0, dtype=torch.long, device=device)
    all_targets = torch.empty(0, dtype=torch.long, device=device)

    model.eval()
    with torch.no_grad():
        for x, cat_x, labels in data_loader:
            x, cat_x, labels = x.to(device), cat_x.to(device), labels.to(device)
            output = model(x, cat_x).squeeze()

            if len(output.shape) == 1:
                pred_ = output.max(0, keepdim=True)[1]
            else:
                pred_ = output.max(1, keepdim=True)[1]

            all_preds = torch.cat([all_preds, pred_])
            all_targets = torch.cat([all_targets, labels])

    accuracy = calc_accuracy(all_preds, all_targets)
    return accuracy

#### Zapis do pliku


In [243]:
def Save_Final_Predictions_To_File(model, final_test_loader):
    model.eval()
    output_array = []
    for x, cat_x in final_test_loader:
        x, cat_x = x.to(device), cat_x.to(device)
        out = model(x, cat_x)
        output_array.append(out)

    pd.Series(output_array).to_csv("pred.csv", index=False, header=False)


### Tworzenie DataLoaderów


In [244]:
train_data = Get_Train_Data()
train_numerical_values, train_categorical_values = Get_Train_Values(train_data)
train_loader, validate_loader = Get_Train_And_Validate_Loaders(train_numerical_values, train_categorical_values, len(train_data))

final_test_data= Get_Final_Test_Data()
final_test_numerical_values, final_test_categorical_values = Get_Final_Test_Values(final_test_data)
final_test_loader = Get_Final_Test_Loader(final_test_numerical_values, final_test_categorical_values, len(final_test_data))

Train data size:  4124
torch.Size([2934, 10])
torch.Size([2934, 23])
torch.Size([2934])
Final test data size:  1767


In [245]:
input_size = train_numerical_values.shape[1] + train_categorical_values.shape[1] - 1
print(input_size)

33


### Trenowanie


In [246]:
class Price_classifier(nn.Module):
    def __init__(self):
        super(Price_classifier, self).__init__()
        self.layer1 = nn.Linear(input_size, 40)
        self.act_1 = nn.LeakyReLU()
        self.d1 = nn.Dropout(0.4)
        self.layer2 = nn.Linear(40, 20)
        self.act_2 = nn.LeakyReLU()
        self.d2 = nn.Dropout(0.4)
        self.layer3 = nn.Linear(20, 3)
    def forward(self, x, cat_x):
        x = torch.cat([x,cat_x],dim=1) # Tu było dim=1
        activation1 = self.act_1(self.layer1(x))
        activation1 = self.d1(activation1)
        activation2 = self.act_2(self.layer2(activation1))
        activation2 = self.d1(activation2)
        output = self.layer3(activation2)

        return output

In [248]:
criterion = nn.CrossEntropyLoss()
model = Price_classifier().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=0.001)

iters = []
losses = []
train_acc = []
val_acc = []
for n in range(100):
    epoch_losses = []
    model.train()
    for x, cat_x, labels in iter(train_loader):
        optimizer.zero_grad()
        x, cat_x, labels = x.to(device), cat_x.to(device), labels.to(device)
        out = model(x, cat_x)
        loss = criterion(out, labels)
        loss.backward()
        epoch_losses.append(loss.item())
        optimizer.step()

    loss_mean = np.array(epoch_losses).mean()
    iters.append(n)
    losses.append(loss_mean)
    test_acc = Get_Accuracy(model, validate_loader)
    print(f"Epoch {n} loss {loss_mean:.3} test_acc: {test_acc:.3}")

    train_acc.append(Get_Accuracy(model, train_loader)) # compute training accuracy
    val_acc.append(test_acc)  # compute validation accuracy


print("Final Training Accuracy: {}".format(train_acc[-1]))
print("Final Validation Accuracy: {}".format(val_acc[-1]))

Epoch 0 loss 11.3 test_acc: 0.641
Epoch 1 loss 1.74 test_acc: 0.53
Epoch 2 loss 1.07 test_acc: 0.466
Epoch 3 loss 0.814 test_acc: 0.42
Epoch 4 loss 0.738 test_acc: 0.356
Epoch 5 loss 0.692 test_acc: 0.479
Epoch 6 loss 0.679 test_acc: 0.395
Epoch 7 loss 0.644 test_acc: 0.526
Epoch 8 loss 0.627 test_acc: 0.469
Epoch 9 loss 0.608 test_acc: 0.521
Epoch 10 loss 0.585 test_acc: 0.5
Epoch 11 loss 0.56 test_acc: 0.545
Epoch 12 loss 0.578 test_acc: 0.5
Epoch 13 loss 0.563 test_acc: 0.521
Epoch 14 loss 0.548 test_acc: 0.675
Epoch 15 loss 0.555 test_acc: 0.557
Epoch 16 loss 0.529 test_acc: 0.658
Epoch 17 loss 0.54 test_acc: 0.575
Epoch 18 loss 0.537 test_acc: 0.654
Epoch 19 loss 0.526 test_acc: 0.666
Epoch 20 loss 0.513 test_acc: 0.601
Epoch 21 loss 0.519 test_acc: 0.492
Epoch 22 loss 0.496 test_acc: 0.557
Epoch 23 loss 0.491 test_acc: 0.612
Epoch 24 loss 0.486 test_acc: 0.478
Epoch 25 loss 0.498 test_acc: 0.589
Epoch 26 loss 0.486 test_acc: 0.666
Epoch 27 loss 0.471 test_acc: 0.539
Epoch 28 loss